#### ENRICH `ORDER_PAYMENTS` FROM BRONZE TO SILVER

In [0]:
import uuid, traceback
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
CATALOG = 'ORDERS_SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'ORDER_PAYMENTS_ENRICHED'
TARGET = f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}"
SOURCE = f"{CATALOG}.BRONZE.ORDERS_PAYMENTS"
AUDIT_TABLE = f"{CATALOG}.AUDIT.PIPELINE_AUDIT_LOG"

In [0]:
order_payments_df = spark.table(SOURCE)
order_payments_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

order_payments_trans_df = order_payments_df.withColumn(
    "payments_norm",
    when(
        (col("payment_installments") > 0) & col("payment_value").isNotNull(),
        round(col("payment_value") / col("payment_installments"), 2)
    ).otherwise(None)
)
order_payments_trans_df.limit(1).display()

In [0]:
order_payments_trans_ordered_df = order_payments_trans_df.select(
    "order_id",
    "payment_sequential",
    "payment_type",
    col("payment_installments").cast("int").alias("payment_installments"),
    "payment_value",
    "payments_norm",
    "file_name",
    "file_path",
    "load_timestamp"
)
order_payments_trans_ordered_df.limit(1).display()

In [0]:
order_payments_trans_ordered_df.createOrReplaceTempView('order_payments_trans_ordered_df_temp_vw')

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.ORDER_PAYMENTS_ENRICHED`

In [0]:
from pyspark.sql.functions import *

def write_audit(run_id, status, start_time, rows=None, error=None):
  end = datetime.now()
  row = (run_id, "ORDER_PAYMENTS_ENRICHED_LOAD", "SILVER", SOURCE, TARGET, status, rows,
           start_time, end, (end - start_time).total_seconds(),
           error[:4000] if error else None,
           spark.sql("SELECT current_user()").first()[0])
  (spark.createDataFrame([row], spark.table(AUDIT_TABLE).schema)
          .write.mode("append").format("delta").saveAsTable(AUDIT_TABLE))

run_id, start_time = str(uuid.uuid4()), datetime.now()
status, rows_written, err = "SUCCESS", None, None

try:
  result = spark.sql(f'''MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY load_timestamp DESC
           ) rn
    FROM order_payments_trans_ordered_df_temp_vw
  )
  WHERE rn = 1
) src
ON (src.order_id = tgt.order_id)
WHEN MATCHED
    THEN UPDATE SET
    tgt.order_id = src.order_id,
    tgt.payment_type = src.payment_type
WHEN NOT MATCHED THEN
INSERT (order_id, payment_sequential, payment_type, payment_installments, payment_value, payments_norm, origin_file_path,
load_timestamp)
VALUES (src.order_id, src.payment_sequential, src.payment_type, src.payment_installments, src.payment_value, src.payments_norm, src.file_path, current_timestamp)
''')
  
  m = result.first().asDict()
  inserted = m.get("num_inserted_rows") or 0
  updated  = m.get("num_updated_rows") or 0
  rows_written = int(inserted + updated)
  print(f"inserted={inserted}, updated={updated}")

except Exception:
  status, err = "FAILED", traceback.format_exc()
  raise
finally:
  write_audit(run_id, status, start_time, rows_written, err)

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.ORDER_PAYMENTS_ENRICHED')